# Facial Valence Detection

Notebook presentation for the **ratemyhuman** project — a three-stage facial valence
detection pipeline: MTCNN face detection → ViT 7-class emotion inference → 3-class valence mapping.

In [ ]:
import os
os.chdir(r"C:\acidvuca\ratemyhuman")
print(f"Working directory: {os.getcwd()}")

## 1  Data gathering

Download the FER-2013 dataset from Kaggle (requires API credentials).

In [ ]:
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"  # to replace with actual credentials
os.environ["KAGGLE_KEY"] = "your_api_key"               # to replace with actual credentials
!kaggle datasets list -s "FER-2013 facial expression reorganized" --sort-by votes
!kaggle datasets download pankaj4321/fer-2013-facial-expression-dataset -p data --unzip

The dataset contains ~35,887 grayscale 48×48 images split into `train/`, `val/`, `test/`
with 7 emotion subdirectories each: angry, disgust, fear, happy, neutral, sad, surprise.

## 2  Exploration

### 2.1  Count images per class and split

In [ ]:
from pathlib import Path

data_dir = Path(r"C:\acidvuca\ratemyhuman\data")
for split in ["train", "test"]:
    split_dir = data_dir / split
    for emotion_dir in sorted(split_dir.iterdir()):
        if emotion_dir.is_dir():
            count = len(list(emotion_dir.glob("*.png")))
            print(f"{split:>5s} | {emotion_dir.name:<10s} | {count:>5,}")

### 2.2  Run the exploration module

Produces distribution plots, sample grids, and an integrity report.

In [ ]:
from ratemyhuman.explore import run_exploration
run_exploration()

Outputs saved to `docs/`: `class_distribution.png`, `valence_distribution.png`, `sample_grid_train.png`, `sample_grid_test.png`.

## 3  GPU environment verification

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv,noheader

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

The `pyproject.toml` uses the nightly PyTorch index (`cu128`), `prerelease = "allow"`,
`unsafe-best-match`, and overrides for facenet-pytorch's stale dependency pins.

## 4  Integrate pretrained model

The pretrained ViT emotion model (`trpakov/vit-face-expression`) is a Vision Transformer
fine-tuned on FER2013 for 7-class emotion recognition (85.8M params, ~71% test accuracy).
It outputs softmax probabilities for: angry, disgust, fear, happy, neutral, sad, surprise.

In [ ]:
!uv add transformers

### 4.1  Initialise the detector

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
from ratemyhuman.model import ValenceDetector
detector = ValenceDetector()

### 4.2  Classify a single image

In [ ]:
import glob
img_path = glob.glob(r"C:\acidvuca\ratemyhuman\data\test\happy\*.png")[0]
result = detector.classify(img_path)
print(result)
print(f"Emotion breakdown: { {k: round(v, 4) for k, v in result.emotion_scores.items()} }")
print(f"Valence breakdown: { {k: round(v, 4) for k, v in result.valence_scores.items()} }")

### 4.3  Spot-check across multiple emotion classes

In [ ]:
for emotion in ["angry", "sad", "fear", "surprise", "neutral"]:
    imgs = glob.glob(rf"C:\acidvuca\ratemyhuman\data\test\{emotion}\*.png")
    if imgs:
        try:
            result = detector.classify(imgs[0])
            print(f"{emotion:>10s} -> {result}")
        except ValueError as e:
            print(f"{emotion:>10s} -> {e}")

**Note:** MTCNN may fail on some 48×48 FER2013 images (designed for higher-res input).
The neutral misclassification reflects FER2013's known label noise (~65% inter-annotator agreement).
These are single-image spot checks showing the pipeline works end-to-end;
systematic quality is measured in step 6 (validation).

## 5  Build valence mapping layer

The valence mapping (concept note §2.3) is the single source of truth in `model.py`:
- **Negative** = angry + disgust + fear + sad
- **Neutral** = neutral
- **Positive** = happy + surprise

### 5.1  Probability aggregation (inference-time)

In [ ]:
import numpy as np
from ratemyhuman.model import ValenceDetector
fake_probs = np.array([0.01, 0.01, 0.01, 0.90, 0.03, 0.01, 0.03])
result = ValenceDetector.map_to_valence(fake_probs)
print(result)
print(f"Valence scores: {result.valence_scores}")

### 5.2  Ground-truth label mapping (FER2013 folder names → valence)

In [ ]:
from pathlib import Path
from ratemyhuman.model import map_label_to_valence
data_dir = Path(r"C:\acidvuca\ratemyhuman\data\test")
for folder in sorted(data_dir.iterdir()):
    if folder.is_dir():
        valence = map_label_to_valence(folder.name)
        count = len(list(folder.glob("*.png")))
        print(f"{folder.name:>10s} -> {valence:<10s} ({count:,} images)")

The function is case-insensitive and strips whitespace.
All constants (`VALENCE_MAP`, `VALENCE_ORDER`, `VALENCE_COLOURS`, `EMOTION_PALETTE`)
are defined in `model.py` and imported by all other modules — no duplication.

## 6  Validation

The validation module (`validate.py`) implements the process from concept note §4.4:
load dataset → map ground truth → batch inference → compute metrics → generate plots.
Metrics are computed using scikit-learn.

### 6.1  Run the full validation on the FER2013 test set (3,589 images)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
from ratemyhuman.validate import run_validation
report = run_validation()

590 images (16.4%) were skipped because MTCNN could not detect a face in the 48×48 images.
Of the 2,999 successfully processed images:
- **Accuracy (82.86%)** far exceeds random (33.3%) and majority (44.2%) baselines
- **Positive** class performs best (F1 = 0.897); **Neutral** is hardest (F1 = 0.680)
- **MCC of 0.7277** confirms strong multi-class discrimination

### 6.2  Confusion matrix heatmap

In [ ]:
from IPython.display import Image, display
display(Image(filename="docs/confusion_matrix.png"))

### 6.3  Misclassified samples grid

In [ ]:
display(Image(filename="docs/misclassified_samples.png"))

### 6.4  Programmatic access to the validation API

In [ ]:
from ratemyhuman.model import ValenceDetector
from ratemyhuman.validate import ValidationRunner
detector = ValenceDetector()
runner = ValidationRunner(detector, r"C:\acidvuca\ratemyhuman\data\test")
report = runner.run_validation()
print(f"Accuracy: {report.accuracy:.1%}")
print(f"Confusion matrix:\n{report.confusion_matrix}")
print(f"Misclassified examples: {len(report.misclassified)}")

## 7  Tests

In [ ]:
!uv run pytest tests/ -v -m "not slow"

In [ ]:
!uv run pytest tests/ -v -m "slow"